In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.4 Transformer block

> 🎯 **Goal:** Combine the residual stream, pre-norm, attention, and the MLP into
> one reusable block, the unit PocketLM stacks N times.

**Why this matters.** The last three lessons each gave you one ingredient. This is
where they snap together into the actual building block of the model. Once you have
one block that takes a sequence in and returns the same shape out, you have the
whole transformer, because the model is literally just a stack of identical blocks.

**The intuition.** Think of the block as one Lego brick with a fixed shape of studs
on top and matching holes underneath. Because what comes out fits exactly what goes
in, you can click as many bricks together as you like into a tall tower. Each brick
does the same two things in order: let the tokens talk (attention), then let each
token think (MLP), each wrapped in pre-norm and added to the stream.

**The mechanics.** A PocketLM `Block` runs two residual sublayers in sequence:
`x = x + attn(norm1(x))` then `x = x + mlp(norm2(x))`. The attention here is
**causal**, each position may attend only to itself and earlier positions, never
the future, which is what makes the model able to predict the next token. Each
sublayer is pre-normed (l4.2) and added back to the residual stream (l4.1), and the
MLP is the position-wise feed-forward from l4.3. Note the two norms are separate
modules (`norm1`, `norm2`) with their own learned parameters. Because input and
output shapes match, the block is *composable*: feed one block's output straight
into the next.

In [ ]:
from pocketlm.model import Block, PocketLMConfig

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=11, block_size=8, n_layer=1, n_head=2, n_embd=16)
block = Block(cfg).eval()
x = torch.randn(1, 5, 16)
out = block(x)
print("block in", tuple(x.shape), "-> out", tuple(out.shape),
      "(shape preserved, so blocks stack)")

**What just happened.** We built one real `Block` from `pocketlm.model`, fed it a
`(1, 5, 16)` tensor (1 sequence, 5 tokens, 16-dim embeddings), and got `(1, 5, 16)`
back. Same shape in, same shape out, which is the property that lets blocks stack.
We called `.eval()` first so any dropout is switched off and the run is
deterministic. The assert confirms the shape contract that the whole stacked
architecture depends on.

Why stacking buys you so much: depth is where abstraction comes from. Early blocks
work close to the raw characters, later blocks operate on the patterns the earlier
ones discovered, words, phrases, structure. Each block refines the residual stream
a little more.

⚠️ **Common pitfalls**
- Mismatching `n_embd` between the block and the rest of the model. The stream width
  must be identical everywhere or the residual add fails.
- Forgetting `.eval()` when you want reproducible outputs, dropout in train mode
  will make repeated runs differ.
- Assuming more blocks is always better. Depth adds capacity but also cost and
  training difficulty; PocketLM stays deliberately small.

🏋️ **Try it yourself.** Build a tiny stack: create three blocks in a list and run
`for b in blocks: x = b(x)`. Confirm the shape survives all three. Then bump
`n_head` from 2 to 4 in the config (keep `n_embd` divisible by it) and check the
output shape is unchanged, head count is internal to attention and invisible from
outside the block.

In [ ]:
assert tuple(out.shape) == (1, 5, 16)